# Clinical Final Fit — All-Data Model + Imputer + Calibrator

In [1]:
import sys
import json
from datetime import date
import joblib
import numpy as np
import pandas as pd
sys.path.insert(0, r"C:\FYP\src")
from utils.config import (
    TABULAR_CLEAN_PATH, CLINICAL_MODEL_COMPARISON_PATH, CLINICAL_OOF_PREDICTIONS_PATH,
    CHECKPOINTS_CLINICAL_FINAL_DIR, ensure_dirs, RANDOM_SEED,
)
from sklearn.experimental import enable_iterative_imputer  # noqa: F401
from sklearn.impute import IterativeImputer
from sklearn.linear_model import BayesianRidge, LogisticRegression
from sklearn.metrics import roc_auc_score, recall_score
from xgboost import XGBClassifier

ensure_dirs()
pd.set_option("display.width", 120)
print("Imports OK")

Imports OK


## The Winning Model: XGBoost

In [2]:
WINNER_MODEL_NAME = "XGBoost"

model_comparison_df = pd.read_csv(CLINICAL_MODEL_COMPARISON_PATH)
winner_rows = model_comparison_df[model_comparison_df.model == WINNER_MODEL_NAME].set_index("scheme")

print(f"Winning model: {WINNER_MODEL_NAME}")
print("\nSupporting numbers (recall = sensitivity, the metric that decided this):")
print(model_comparison_df.set_index(["scheme", "model"])[["recall_mean", "auc_mean"]])

Winning model: XGBoost

Supporting numbers (recall = sensitivity, the metric that decided this):
                                       recall_mean  auc_mean
scheme             model                                    
Repeated 5x20 CV   XGBoost                  0.7433    0.9077
                   LogisticRegression       0.6157    0.8786
                   RandomForest             0.6920    0.9068
Cohort-out         XGBoost                  0.6757    0.8375
                   LogisticRegression       0.7027    0.8793
                   RandomForest             0.7568    0.8901
Leave-one-site-out XGBoost                  0.7783    0.8195
                   LogisticRegression       0.7483    0.8221
                   RandomForest             0.7290    0.8324


## Load `tabular_clean.csv`

In [3]:
FEATURES = ["creatinine", "LYVE1", "REG1B", "TFF1", "plasma_CA19_9", "age", "sex"]
METADATA_COLS = ["sample_id", "patient_cohort", "sample_origin", "stage", "diagnosis"]
TARGET_COLS = ["dx", "target_binary"]

tabular_clean_df = pd.read_csv(TABULAR_CLEAN_PATH)
feature_matrix = tabular_clean_df[FEATURES].copy()
METADATA = tabular_clean_df[METADATA_COLS].copy()
TARGETS = tabular_clean_df[TARGET_COLS].copy()

print(f"feature_matrix: {feature_matrix.shape}   METADATA: {METADATA.shape}   TARGETS: {TARGETS.shape}")
print(f"plasma_CA19_9 missing: {feature_matrix['plasma_CA19_9'].isna().sum()}")

feature_matrix: (590, 7)   METADATA: (590, 5)   TARGETS: (590, 2)
plasma_CA19_9 missing: 240


## The Settled Imputer: `MICE_CA19_9Imputer`

In [4]:
class MICE_CA19_9Imputer:
    PREDICTORS = ["creatinine", "LYVE1", "REG1B", "TFF1", "age"]
    TARGET = "plasma_CA19_9"

    def __init__(self, random_state=RANDOM_SEED):
        self.imputer = IterativeImputer(estimator=BayesianRidge(), random_state=random_state)

    def fit(self, train_df):
        self.imputer.fit(train_df[self.PREDICTORS + [self.TARGET]])
        return self

    def transform(self, target_df):
        out = target_df.copy()
        out[self.TARGET] = self.imputer.transform(out[self.PREDICTORS + [self.TARGET]])[:, -1]
        return out


print("MICE_CA19_9Imputer defined.")

MICE_CA19_9Imputer defined.


## All-Data Fit — Imputer + Model, No Folds

In [5]:
final_imputer = MICE_CA19_9Imputer().fit(feature_matrix)
feature_matrix_imputed = final_imputer.transform(feature_matrix)
assert feature_matrix_imputed["plasma_CA19_9"].isna().sum() == 0

final_model = XGBClassifier(n_estimators=100, max_depth=3, eval_metric="logloss", random_state=0)
final_model.fit(feature_matrix_imputed[FEATURES], TARGETS["target_binary"])

print(f"Final imputer fit on all {len(feature_matrix)} patients.")
print(f"Final model ({WINNER_MODEL_NAME}) fit on all {len(feature_matrix_imputed)} patients, {TARGETS['target_binary'].sum()} PDAC / {(~TARGETS['target_binary'].astype(bool)).sum()} not-PDAC.")

Final imputer fit on all 590 patients.
Final model (XGBoost) fit on all 590 patients, 199 PDAC / 391 not-PDAC.


## Calibrator — Platt Scaling, Fit on Saved Out-of-Fold Predictions

In [6]:
oof_df = pd.read_csv(CLINICAL_OOF_PREDICTIONS_PATH)
winner_oof = oof_df[oof_df.model == WINNER_MODEL_NAME]
assert len(winner_oof) > 0, f"No OOF rows found for {WINNER_MODEL_NAME}"

X_calib = winner_oof[["y_proba"]].values
y_calib = winner_oof["y_true"].values

calibrator = LogisticRegression()
calibrator.fit(X_calib, y_calib)

print(f"Calibrator (Platt scaling) fit on {len(winner_oof):,} {WINNER_MODEL_NAME} OOF rows "
      f"({winner_oof['sample_id'].nunique()} unique patients, {int(winner_oof['repeat_idx'].max()) + 1} repeats each).")

Calibrator (Platt scaling) fit on 11,800 XGBoost OOF rows (590 unique patients, 20 repeats each).


## Sanity Check — In-Sample Only, Not New Performance Evidence

In [7]:
in_sample_proba = final_model.predict_proba(feature_matrix_imputed[FEATURES])[:, 1]
in_sample_preds = final_model.predict(feature_matrix_imputed[FEATURES])
in_sample_auc = roc_auc_score(TARGETS["target_binary"], in_sample_proba)
in_sample_recall = recall_score(TARGETS["target_binary"], in_sample_preds)

cv_auc = winner_rows.loc["Repeated 5x20 CV", "auc_mean"]
cv_recall = winner_rows.loc["Repeated 5x20 CV", "recall_mean"]

print("IN-SAMPLE (fit-on-everything, seen-every-label) -- NOT a performance estimate:")
print(f"  AUC:    {in_sample_auc:.4f}")
print(f"  Recall: {in_sample_recall:.4f}")
print(f"\nFor comparison, the REAL validated estimate (repeated 5x20 CV, held-out folds):")
print(f"  AUC:    {cv_auc:.4f}")
print(f"  Recall: {cv_recall:.4f}")
print(f"\nIn-sample looks better, as expected -- this gap is exactly why in-sample numbers "
      f"are never reported as evidence of real-world performance.")

IN-SAMPLE (fit-on-everything, seen-every-label) -- NOT a performance estimate:
  AUC:    1.0000
  Recall: 1.0000

For comparison, the REAL validated estimate (repeated 5x20 CV, held-out folds):
  AUC:    0.9077
  Recall: 0.7433

In-sample looks better, as expected -- this gap is exactly why in-sample numbers are never reported as evidence of real-world performance.


## Save to `checkpoints/clinical/final/`

In [8]:
joblib.dump(final_model, CHECKPOINTS_CLINICAL_FINAL_DIR / "model.pkl")
joblib.dump(final_imputer, CHECKPOINTS_CLINICAL_FINAL_DIR / "ca19_9_imputer.pkl")
joblib.dump(calibrator, CHECKPOINTS_CLINICAL_FINAL_DIR / "calibrator.pkl")

model_card = {
    "model": "xgboost",
    "model_hyperparameters": {
        "n_estimators": 100, "max_depth": 3, "eval_metric": "logloss", "random_state": 0,
    },
    "trained_on": date.today().isoformat(),
    "n_patients": int(len(feature_matrix)),
    "n_pdac": int(TARGETS["target_binary"].sum()),
    "n_not_pdac": int((~TARGETS["target_binary"].astype(bool)).sum()),
    "imputer": "MICE_CA19_9Imputer",
    "comparison_metrics_that_justified_this_model": {
        scheme: {
            model: {
                "recall_mean": float(model_comparison_df.set_index(["scheme", "model"]).loc[(scheme, model), "recall_mean"]),
                "auc_mean": float(model_comparison_df.set_index(["scheme", "model"]).loc[(scheme, model), "auc_mean"]),
            }
            for model in ["XGBoost", "LogisticRegression", "RandomForest"]
        }
        for scheme in ["Repeated 5x20 CV", "Cohort-out", "Leave-one-site-out"]
    },
    "selection_reasoning": (
        "XGBoost has the best sensitivity/recall -- the clinically prioritised metric, since "
        "missing a PDAC case is costlier than a false alarm -- in 2 of 3 CV schemes (repeated "
        "5x20 CV and leave-one-site-out), with AUC negligibly different from Random Forest in "
        "those same two schemes. Random Forest only wins the single, non-repeated cohort-out "
        "split, the statistically weakest of the three estimates. See "
        "docs/Tabular_Model_Comparison_documentation.md Final Summary for full reasoning."
    ),
    "calibrator": "Platt scaling (LogisticRegression on raw predict_proba)",
    "calibrator_fit_rows": int(len(winner_oof)),
    "calibrator_fit_unique_patients": int(winner_oof["sample_id"].nunique()),
    "calibrator_reasoning": (
        "Platt scaling chosen over isotonic regression: isotonic typically needs thousands of "
        "samples to avoid overfitting/staircase artifacts, and the effective sample size here "
        "is 590 unique patients (not the 11,800 rows in the OOF file), favouring the simpler, "
        "more stable parametric option. Fit on all repeated-CV OOF rows directly (not averaged "
        "to one row per patient first) since the 20 repeats use different fold partners each "
        "time and carry real additional information, not pure duplication."
    ),
    "in_sample_sanity_check": {
        "note": "NOT a performance estimate -- scored on data the model was trained on",
        "auc": float(in_sample_auc), "recall": float(in_sample_recall),
    },
}

with open(CHECKPOINTS_CLINICAL_FINAL_DIR / "model_card.json", "w") as f:
    json.dump(model_card, f, indent=2)

print(f"Saved to {CHECKPOINTS_CLINICAL_FINAL_DIR}:")
for fname in ["model.pkl", "ca19_9_imputer.pkl", "calibrator.pkl", "model_card.json"]:
    fpath = CHECKPOINTS_CLINICAL_FINAL_DIR / fname
    print(f"  {fname}: {'exists, ' + str(fpath.stat().st_size) + ' bytes' if fpath.exists() else 'MISSING'}")

Saved to C:\FYP\checkpoints\clinical\final:
  model.pkl: exists, 116590 bytes
  ca19_9_imputer.pkl: exists, 17587 bytes
  calibrator.pkl: exists, 863 bytes
  model_card.json: exists, 2644 bytes


## Confirm All Four Files Exist

In [9]:
expected_files = ["model.pkl", "ca19_9_imputer.pkl", "calibrator.pkl", "model_card.json"]
all_present = all((CHECKPOINTS_CLINICAL_FINAL_DIR / f).exists() for f in expected_files)
print(f"All 4 output files present: {all_present}")
for f in expected_files:
    print(f"  {'OK ' if (CHECKPOINTS_CLINICAL_FINAL_DIR / f).exists() else 'MISSING'} {f}")

print("\nmodel_card.json contents:")
with open(CHECKPOINTS_CLINICAL_FINAL_DIR / "model_card.json") as f:
    print(json.dumps(json.load(f), indent=2))

All 4 output files present: True
  OK  model.pkl
  OK  ca19_9_imputer.pkl
  OK  calibrator.pkl
  OK  model_card.json

model_card.json contents:
{
  "model": "xgboost",
  "model_hyperparameters": {
    "n_estimators": 100,
    "max_depth": 3,
    "eval_metric": "logloss",
    "random_state": 0
  },
  "trained_on": "2026-07-16",
  "n_patients": 590,
  "n_pdac": 199,
  "n_not_pdac": 391,
  "imputer": "MICE_CA19_9Imputer",
  "comparison_metrics_that_justified_this_model": {
    "Repeated 5x20 CV": {
      "XGBoost": {
        "recall_mean": 0.7433,
        "auc_mean": 0.9077
      },
      "LogisticRegression": {
        "recall_mean": 0.6157,
        "auc_mean": 0.8786
      },
      "RandomForest": {
        "recall_mean": 0.692,
        "auc_mean": 0.9068
      }
    },
    "Cohort-out": {
      "XGBoost": {
        "recall_mean": 0.6757,
        "auc_mean": 0.8375
      },
      "LogisticRegression": {
        "recall_mean": 0.7027,
        "auc_mean": 0.8793
      },
      "RandomFore